In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Load Phase 1 outputs
train_df = pd.read_csv('data/train_processed_phase1.csv')
test_df = pd.read_csv('data/test_processed_phase1.csv')

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"\nCategorical columns:")
for col in ['protocol_type', 'service', 'flag']:
    print(f"  {col}: {train_df[col].nunique()} unique values → {train_df[col].unique()[:5]}...")

Train: (125973, 44)
Test:  (22544, 44)

Categorical columns:
  protocol_type: 3 unique values → <StringArray>
['tcp', 'udp', 'icmp']
Length: 3, dtype: str...
  service: 70 unique values → <StringArray>
['ftp_data', 'other', 'private', 'http', 'remote_job']
Length: 5, dtype: str...
  flag: 11 unique values → <StringArray>
['SF', 'S0', 'REJ', 'RSTR', 'SH']
Length: 5, dtype: str...


In [2]:
# Drop 'difficulty' — it's metadata about how hard the record is to classify,
# not an actual network feature. Keeping it would be data leakage
# (the model would be "cheating" by using info it wouldn't have in real life)

# Drop 'label' — we'll use 'attack_category' instead (Normal/DoS/Probe/R2L/U2R)
# The original label (like 'neptune', 'smurf') is too granular for our model
# and some test attacks don't appear in training at all

train_df = train_df.drop(columns=['difficulty', 'label'])
test_df = test_df.drop(columns=['difficulty', 'label'])

print(f"Columns remaining: {train_df.shape[1]}")
print(f"Target column: attack_category")
print(f"Feature columns: {train_df.shape[1] - 1}")

Columns remaining: 42
Target column: attack_category
Feature columns: 41


In [3]:
# ENCODING CATEGORICAL FEATURES
#
# We'll use Label Encoding: convert each text value to a number
# Example: protocol_type → tcp=0, udp=1, icmp=2
#
# Why Label Encoding and not One-Hot Encoding?
# - One-Hot would create a new column for EVERY unique value
# - 'service' alone has 70 unique values → 70 new columns
# - Random Forest handles Label Encoding perfectly because it
#   splits on individual values, not on magnitude
# - If we used a neural network later, we'd reconsider this choice

# Store the encoders so we can reuse them in the prediction pipeline (Phase 5)
encoders = {}
categorical_cols = ['protocol_type', 'service', 'flag']

for col in categorical_cols:
    le = LabelEncoder()

    # Important: fit on BOTH train and test combined
    # Why? The test set might have values not in training
    # (like extra service types). If we only fit on train,
    # the encoder would crash on unseen test values.
    all_values = pd.concat([train_df[col], test_df[col]]).unique()
    le.fit(all_values)

    # Transform both sets
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

    # Save the encoder
    encoders[col] = le

    print(f"{col}:")
    print(f"  {len(le.classes_)} unique values")
    print(f"  Mapping sample: {dict(zip(le.classes_[:5], le.transform(le.classes_[:5])))}")
    print()

print("All categorical columns encoded!")

protocol_type:
  3 unique values
  Mapping sample: {'icmp': np.int64(0), 'tcp': np.int64(1), 'udp': np.int64(2)}

service:
  70 unique values
  Mapping sample: {'IRC': np.int64(0), 'X11': np.int64(1), 'Z39_50': np.int64(2), 'aol': np.int64(3), 'auth': np.int64(4)}

flag:
  11 unique values
  Mapping sample: {'OTH': np.int64(0), 'REJ': np.int64(1), 'RSTO': np.int64(2), 'RSTOS0': np.int64(3), 'RSTR': np.int64(4)}

All categorical columns encoded!


In [4]:
# Separate features and target
# X = everything the model sees (the 41 network features, now all numerical)
# y = what we want the model to predict (attack category)

X_train = train_df.drop(columns=['attack_category'])
y_train = train_df['attack_category']

X_test = test_df.drop(columns=['attack_category'])
y_test = test_df['attack_category']

# Encode the target labels too
target_encoder = LabelEncoder()
target_encoder.fit(['Normal', 'DoS', 'Probe', 'R2L', 'U2R'])
y_train_encoded = target_encoder.transform(y_train)
y_test_encoded = target_encoder.transform(y_test)

print(f"Features shape: {X_train.shape}")
print(f"Target classes: {dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))}")
print(f"\nTarget distribution (train):")
print(y_train.value_counts())

Features shape: (125973, 41)
Target classes: {np.str_('DoS'): np.int64(0), np.str_('Normal'): np.int64(1), np.str_('Probe'): np.int64(2), np.str_('R2L'): np.int64(3), np.str_('U2R'): np.int64(4)}

Target distribution (train):
attack_category
Normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64


In [5]:
# NORMALIZATION
#
# StandardScaler transforms each feature to have mean=0 and std=1
#
# Why? Feature 'src_bytes' ranges from 0 to 1,379,963,888
# while 'serror_rate' ranges from 0 to 1
# Without scaling, large-valued features can dominate distance calculations
#
# Random Forest doesn't strictly NEED this (it splits on thresholds,
# not distances), but we do it anyway because:
# 1. It's good practice you should demonstrate
# 2. If you swap to a neural network or SVM later, you'll need it
# 3. It makes feature comparison plots more readable

scaler = StandardScaler()

# Fit ONLY on training data, then transform both
# Why? Fitting on test data would be "peeking at the future" — data leakage
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),  # transform only, don't fit!
    columns=X_test.columns,
    index=X_test.index
)

# Verify: mean should be ~0, std should be ~1 for training set
print("Training set after scaling (first 5 features):")
print(X_train_scaled.iloc[:, :5].describe().loc[['mean', 'std']].round(2))

Training set after scaling (first 5 features):
      duration  protocol_type  service  flag  src_bytes
mean       0.0           -0.0      0.0   0.0        0.0
std        1.0            1.0      1.0   1.0        1.0


In [7]:
import joblib
import os
os.makedirs('models', exist_ok=True)

# Save the scaled feature matrices and encoded targets
# We save both scaled and unscaled versions — unscaled is useful for debugging

# Scaled data (for model training)
X_train_scaled.to_csv('data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('data/X_test_scaled.csv', index=False)

# Encoded targets
pd.Series(y_train_encoded, name='target').to_csv('data/y_train.csv', index=False)
pd.Series(y_test_encoded, name='target').to_csv('data/y_test.csv', index=False)

# Save the encoders and scaler — we'll need these in Phase 5
# when we build the live prediction pipeline
joblib.dump(encoders, 'models/encoders.joblib')
joblib.dump(scaler, 'models/scaler.joblib')
joblib.dump(target_encoder, 'models/target_encoder.joblib')

print("Saved:")
print("  data/X_train_scaled.csv")
print("  data/X_test_scaled.csv")
print("  data/y_train.csv")
print("  data/y_test.csv")
print("  models/encoders.joblib")
print("  models/scaler.joblib")
print("  models/target_encoder.joblib")
print("\nPhase 2 complete!")

Saved:
  data/X_train_scaled.csv
  data/X_test_scaled.csv
  data/y_train.csv
  data/y_test.csv
  models/encoders.joblib
  models/scaler.joblib
  models/target_encoder.joblib

Phase 2 complete!
